# 03 — AML cluster labels for the other circulating cells

The AML dataset has 13 clusters (PM1–PM9, LM1, LM2, CC1, CC2). This notebook gives every other circulating cell
the AML cluster it resembles most, using the Harmony space of notebook 02.

Steps:
1. **Vote.** Each non-AML cell looks at its 30 nearest AML cells. Their clusters vote, closer cells count more.
2. **Distance cutoff.** A cell that lies too far from the AML cells gets no label ("unassigned").
3. **Refinement.** A cell whose vote was split takes the most common label among its own 30 nearest neighbours
   (all circulating cells), repeated until almost nothing changes.

Output: `labels.tsv.gz`, one row per circulating cell; the final label is the column `aml_label`.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

from settings import AML_CLUSTERS, AML_DATASET, INTEGRATION_FILE, LABEL_TABLE, UNASSIGNED

## Settings

In [ ]:
N_NEIGHBORS = 30                 # neighbours that vote
DISTANCE_QUANTILE = 0.99         # distance cutoff: this quantile of the AML cells' own distances
LOW_SUPPORT = 0.55               # votes with a smaller winning share count as "split"
STOP_FRACTION = 0.001            # refinement stops when fewer than 0.1% of the cells change in a round
MAX_ROUNDS = 50

In [ ]:
integration = ad.read_h5ad(INTEGRATION_FILE)

cells = integration.obs_names
harmony = np.asarray(integration.obsm["X_harmony"])

is_aml = integration.obs["dataset"].eq(AML_DATASET).to_numpy()
aml_cluster = integration.obs.loc[is_aml, "published_label"].astype(str).to_numpy()

print(f"{len(cells):,} circulating cells: {is_aml.sum():,} AML cells, {(~is_aml).sum():,} other cells")

## 1. Vote of the 30 nearest AML cells

`KNeighborsClassifier` (scikit-learn) with `weights="distance"`: each of the 30 nearest AML cells votes for its
own cluster with weight 1 / distance. For every non-AML cell:

- `vote_label`: the cluster with the largest share of the weighted vote;
- `vote_support`: that share (1 = all 30 neighbours agree; it is not a probability).

In [ ]:
classifier = KNeighborsClassifier(n_neighbors=N_NEIGHBORS, weights="distance", n_jobs=-1)
classifier.fit(harmony[is_aml], aml_cluster)

vote_shares = classifier.predict_proba(harmony[~is_aml])

vote_label = classifier.classes_[vote_shares.argmax(axis=1)]
vote_support = vote_shares.max(axis=1)

## 2. Distance cutoff

A cell far away from every AML cell still has 30 "nearest" AML cells, but their vote means little.
The distance of a cell is the distance to its 30th nearest AML cell. For each AML cluster the cutoff is the 99th
percentile of that distance among the cluster's own AML cells (the cell itself is not counted as its neighbour).
A non-AML cell keeps its vote label only if its distance is within the cutoff of that label.

In [ ]:
# kneighbors() without input: every AML cell's neighbours among the other AML cells
aml_distances, _ = classifier.kneighbors(return_distance=True)
aml_distance = aml_distances[:, -1]

cutoff_of_cluster = {}

for cluster in classifier.classes_:
    cutoff_of_cluster[cluster] = np.quantile(aml_distance[aml_cluster == cluster], DISTANCE_QUANTILE)

query_distances, _ = classifier.kneighbors(harmony[~is_aml], return_distance=True)
distance = query_distances[:, -1]
distance_cutoff = np.array([cutoff_of_cluster[label] for label in vote_label])

label_of_query = np.where(distance <= distance_cutoff, vote_label, UNASSIGNED)

In [ ]:
# one label per circulating cell: AML cells keep their own cluster
labels = np.empty(len(cells), dtype=object)
labels[is_aml] = aml_cluster
labels[~is_aml] = label_of_query

support = np.full(len(cells), np.nan)
support[~is_aml] = vote_support

unassigned_percent = 100 * (label_of_query == UNASSIGNED).mean()
print(f"non-AML cells without a label after the distance cutoff: {unassigned_percent:.1f}%")

## 3. Refinement of split votes

Cells with `vote_support` below 0.55 sit between clusters, and their labels look scattered. Each of these cells
takes the most common label among its 30 nearest neighbours in the Harmony space. The neighbours are all
circulating cells (AML cells and other cells) with their current labels; unassigned neighbours do not count.
A cell changes only if another label is strictly more common than its own. Because labels of neighbours change
too, the step is repeated until fewer than 0.1% of the cells change in a round.

AML cells, cells with support ≥ 0.55 and unassigned cells never change.

In [ ]:
# the 30 nearest neighbours of every cell (column 0 is the cell itself)
neighbour_search = NearestNeighbors(n_neighbors=N_NEIGHBORS + 1, n_jobs=-1).fit(harmony)
neighbours = neighbour_search.kneighbors(harmony, return_distance=False)[:, 1:]

# clusters in alphabetical order; with equal counts the first cluster in this order wins
clusters = np.array(sorted(AML_CLUSTERS))
code_of_cluster = {cluster: code for code, cluster in enumerate(clusters)}


def count_neighbour_labels(labels):
    """For every cell: its own label code, and how many of its neighbours carry each cluster label."""
    codes = np.array([code_of_cluster.get(label, -1) for label in labels])      # -1 = unassigned
    neighbour_codes = codes[neighbours]

    counts = np.zeros((len(labels), len(clusters)), dtype=int)

    for code in range(len(clusters)):
        counts[:, code] = (neighbour_codes == code).sum(axis=1)

    return codes, counts

In [ ]:
can_change = ~is_aml & (labels != UNASSIGNED) & (support < LOW_SUPPORT)
refined = labels.copy()

for round_number in range(1, MAX_ROUNDS + 1):
    codes, counts = count_neighbour_labels(refined)

    own_count = counts[np.arange(len(cells)), codes]
    most_common_count = counts.max(axis=1)
    most_common_label = clusters[counts.argmax(axis=1)]

    changes = can_change & (most_common_count > own_count)
    refined[changes] = most_common_label[changes]

    print(f"round {round_number}: {int(changes.sum()):,} cells changed")

    if changes.mean() < STOP_FRACTION:
        break

print(f"{can_change.sum():,} cells had a split vote; {(refined != labels).sum():,} labels changed in total")

## Save

In [ ]:
table = pd.DataFrame(
    {
        "dataset": integration.obs["dataset"].astype(str).to_numpy(),
        "vote_label": pd.Series(vote_label, index=cells[~is_aml]).reindex(cells).to_numpy(),
        "vote_support": support,
        "distance": pd.Series(distance, index=cells[~is_aml]).reindex(cells).to_numpy(),
        "distance_cutoff": pd.Series(distance_cutoff, index=cells[~is_aml]).reindex(cells).to_numpy(),
        "label_before_refinement": labels,
        "aml_label": refined,
    },
    index=cells,
)
table.index.name = "cell_id"

LABEL_TABLE.parent.mkdir(parents=True, exist_ok=True)
table.to_csv(LABEL_TABLE, sep="\t")

In [ ]:
other_cells = table[~is_aml]
display(pd.crosstab(other_cells["label_before_refinement"], other_cells["aml_label"]))

print(f"saved {LABEL_TABLE}")